# Generate circuits with GPT

In [2]:
from pathlib import Path
import os
import pickle
from contextlib import nullcontext
import torch
import tiktoken
from nanoGPT.model import GPTConfig, GPT
import pandas as pd
import json
from tqdm import tqdm
import random
import numpy as np
from matplotlib import pyplot as plt

In [16]:
out_dir = '/home/ityagin/GPT_stuff/nanoGPT/out-adapt-gpt_n12w_qaoa_090724/'
data_dir = '/home/ityagin/GPT_stuff/nanoGPT/data/n12w_qaoa_090724/'

In [9]:
seed = 1337
init_from = 'resume' # either 'resume' (from an out_dir) or a gpt2 variant (e.g. 'gpt2-xl')
device = 'cuda' # examples: 'cpu', 'cuda', 'cuda:0', 'cuda:1', etc.
dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16' # 'float32' or 'bfloat16' or 'float16'
compile = False # use PyTorch 2.0 to compile the model to be faster
#exec(open(nanogpt_path.joinpath('configurator.py')).read()) # overrides from command line or config file
# -----------------------------------------------------------------------------

torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cuda.matmul.allow_tf32 = True # allow tf32 on matmul
torch.backends.cudnn.allow_tf32 = True # allow tf32 on cudnn
device_type = 'cuda' if 'cuda' in device else 'cpu' # for later use in torch.autocast
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]
ctx = nullcontext() if device_type == 'cpu' else torch.amp.autocast(device_type=device_type, dtype=ptdtype)

In [10]:
# init from a model saved in a specific directory
out_path = Path(out_dir)
ckpt_path = f'{out_dir}/ckpt.pt'
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
model = GPT(gptconf)
state_dict = checkpoint['model']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
model.load_state_dict(state_dict)

number of parameters: 11.52M


<All keys matched successfully>

In [11]:
model.eval()
model.to(device)
print()

In [17]:
meta = pd.read_pickle(f'{data_dir}/meta.pkl')

In [18]:
stoi, itos = meta['stoi'], meta['itos']
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: [itos[i] for i in l]

In [19]:
def extract_graph(token_seq):
    graph_seq = []

    for idx, tok in enumerate(token_seq):
        graph_seq.append(tok)
        if tok == 'end_of_graph':
            break
    adapt_seq = token_seq[idx+1:-1]
    return graph_seq, adapt_seq

## Start generation

In [20]:
num_samples = 5 # number of samples to draw
max_new_tokens = 500 # number of tokens generated in each sample
temperature = 0.1 # 1.0 = no change, < 1.0 = less random, > 1.0 = more random, in predictions
top_k = 200 # retain only the top_k most likely tokens, clamp others to have 0 probability

### Opening val dataset

In [79]:
sample_size = 300

In [33]:
all_data_df = pd.read_pickle(
    f'{data_dir}/combined_res_tok_shf_df.pkl'
)

print("Dataset size:", len(all_data_df), 'circuits.')

Dataset size: 36030 circuits.


In [34]:
print(all_data_df['token_seq_round_d2'][264])

['bos', (2, 3), 0.68, (3, 4), 0.98, (1, 6), 0.76, (4, 6), 0.19, (5, 6), 0.57, (1, 7), 0.33, (4, 7), 0.17, (2, 8), 0.82, (3, 8), 0.08, (4, 8), 0.67, (6, 8), 0.63, (1, 9), 0.5, (2, 9), 0.14, (5, 9), 0.81, (6, 9), 0.54, (8, 9), 0.86, (3, 10), 0.3, (6, 10), 0.26, (7, 10), 0.49, (8, 10), 0.94, (1, 11), 0.67, (2, 11), 0.07, (3, 11), 0.78, (6, 11), 0.05, (7, 11), 0.1, (8, 11), 0.89, (2, 12), 0.81, (4, 12), 0.77, (5, 12), 0.85, (6, 12), 0.25, (7, 12), 0.07, (9, 12), 0.35, 'end_of_graph', 'new_layer_p', 101, -0.79, -0.0, 'new_layer_p', 245, -0.79, 0.0, 'new_layer_p', 200, -0.79, -0.0, 'new_layer_p', 256, -0.79, -0.0, 'new_layer_p', 168, -0.79, 0.0, 'new_layer_p', 273, -0.79, -0.0, 'new_layer_p', 72, -0.79, 0.0, 'new_layer_p', 56, -0.79, -0.0, 'new_layer_p', 93, -0.79, 0.0, 'new_layer_p', 173, -0.79, -0.0, 'new_layer_p', 99, 0.79, 0.43, 'eos']


In [80]:
val_df = all_data_df[
    all_data_df['label'] == 'val'
]

token_seq_col = 'token_seq_round_d2'

sampled_idx_list = random.sample(list(val_df.index), sample_size)

In [81]:
# BATCHING wrt num_samples

adapt_gpt_val_samples_list = []

for graph_idx in tqdm(sampled_idx_list):
    graph_df_row = val_df.loc[graph_idx]
    start, adapt_seq = extract_graph(val_df[token_seq_col].loc[graph_idx])
    start_ids = encode(start)
    x = (torch.tensor(start_ids, dtype=torch.long, device=device)[None, ...])

    adapt_gpt_out_dict = dict()
    adapt_gpt_out_dict['graph'] = start[1:-1]
    adapt_gpt_out_dict['q_circuits'] = []
    adapt_gpt_out_dict['adapt_circuit'] = adapt_seq
    adapt_gpt_out_dict['adapt_full_ar'] = graph_df_row['approx_ratio']
    adapt_gpt_out_dict['graph_prefix'] = graph_df_row['graph_id']
    adapt_gpt_out_dict['energy_mqlib'] = graph_df_row['energy_mqlib']
    
    # run generation
    with torch.no_grad():
        with ctx:
            y = model.generate(
                    x.repeat(num_samples, 1),
                    max_new_tokens,
                    temperature=temperature,
                    top_k=top_k
                )
            for k in range(num_samples):
                cur_gen_result = decode(y[k].tolist())
                cur_circ = []
                circ_flag = 0
                for idx, tok in enumerate(cur_gen_result):
                    if tok == 'end_of_graph':
                        circ_flag = 1
                    if circ_flag:
                        cur_circ.append(tok)
                    if tok == 'eos':
                        break
                adapt_gpt_out_dict['q_circuits'].append(cur_circ[1:-1])

    adapt_gpt_val_samples_list.append(adapt_gpt_out_dict)

100%|█████████████████████████████████████████████████████████████████████████████| 300/300 [07:06<00:00,  1.42s/it]


## Formatting for `ADAPT.jl`

In [82]:
def circ_sanity_check(cur_q_circ):
    
    lr_sep_list = cur_q_circ[0::4]
    op_idx_list = cur_q_circ[1::4]

    num_vals = cur_q_circ[2::4] + cur_q_circ[3::4]

    if any(
        [type(el) != int for el in op_idx_list]
    ):
        print('wrong op_idx_list')
        return False

    if any(
        [type(el) != str for el in lr_sep_list]
    ):
        print('wrong lr_sep_list')
        return False

    return True

In [83]:
for idx in range(len(adapt_gpt_val_samples_list)):
    adapt_gpt_val_samples_list[idx]['q_circuits'] = (
        list(
            filter(
                circ_sanity_check,
                adapt_gpt_val_samples_list[idx]['q_circuits']
            )
        )
    )

for gr_dict in adapt_gpt_val_samples_list:
    graph_jl_list = []

    graph_edges_list = gr_dict['graph'][::2]
    graph_weights_list = gr_dict['graph'][1::2]
    for edge_idx, edge in enumerate(graph_edges_list):
        cur_edge = list(edge)
        cur_edge += [graph_weights_list[edge_idx]]
        graph_jl_list.append(cur_edge)

    gr_dict['graph_w_jl'] = graph_jl_list

In [84]:
pd.DataFrame(adapt_gpt_val_samples_list)

,graph,q_circuits,adapt_circuit,adapt_full_ar,graph_prefix,energy_mqlib,graph_w_jl
0,"[(1, 2), 0.11, (1, 3), 0.76, (2, 3), 0.94, (1,...","[[new_layer_p, 277, -0.79, 0.0, new_layer_p, 2...","[new_layer_p, 276, -0.79, -0.0, new_layer_p, 2...",1.000000,n_12_worker_unknown_pid_1233158_ts_24-09-09__0...,-20.29,"[[1, 2, 0.11], [1, 3, 0.76], [2, 3, 0.94], [1,..."
1,"[(1, 3), 0.4, (3, 5), 0.15, (4, 5), 0.75, (4, ...","[[new_layer_p, 152, -0.79, 0.0, new_layer_p, 1...","[new_layer_p, 152, -0.79, 0.0, new_layer_p, 14...",1.000000,n_12_worker_unknown_pid_1233167_ts_24-09-09__0...,-15.63,"[[1, 3, 0.4], [3, 5, 0.15], [4, 5, 0.75], [4, ..."
2,"[(1, 2), 0.58, (2, 3), 0.76, (1, 4), 0.67, (3,...","[[new_layer_p, 48, -0.79, 0.0, new_layer_p, 25...","[new_layer_p, 48, -0.79, -0.0, new_layer_p, 25...",1.000000,n_12_worker_unknown_pid_1258282_ts_24-09-10__1...,-16.52,"[[1, 2, 0.58], [2, 3, 0.76], [1, 4, 0.67], [3,..."
3,"[(1, 3), 0.08, (2, 3), 0.39, (1, 4), 0.86, (1,...","[[new_layer_p, 36, -0.79, 0.0, new_layer_p, 22...","[new_layer_p, 37, -0.79, -0.0, new_layer_p, 14...",1.000000,n_12_worker_unknown_pid_1233104_ts_24-09-09__0...,-14.72,"[[1, 3, 0.08], [2, 3, 0.39], [1, 4, 0.86], [1,..."
4,"[(1, 2), 0.18, (2, 3), 0.71, (1, 5), 0.54, (3,...","[[new_layer_p, 173, -0.79, 0.0, new_layer_p, 2...","[new_layer_p, 173, -0.79, -0.0, new_layer_p, 1...",1.000000,n_12_worker_unknown_pid_1233017_ts_24-09-09__0...,-21.19,"[[1, 2, 0.18], [2, 3, 0.71], [1, 5, 0.54], [3,..."
...,...,...,...,...,...,...,...
295,"[(2, 4), 0.56, (1, 5), 0.73, (3, 5), 0.33, (4,...","[[new_layer_p, 224, -0.79, 0.0, new_layer_p, 1...","[new_layer_p, 225, -0.79, -0.01, new_layer_p, ...",0.996809,n_12_worker_unknown_pid_1233065_ts_24-09-09__0...,-14.37,"[[2, 4, 0.56], [1, 5, 0.73], [3, 5, 0.33], [4,..."
296,"[(1, 3), 0.87, (1, 4), 0.87, (2, 4), 0.89, (1,...","[[new_layer_p, 53, -0.79, 0.0, new_layer_p, 26...","[new_layer_p, 53, -0.79, 0.0, new_layer_p, 41,...",1.000000,n_12_worker_unknown_pid_1258257_ts_24-09-10__1...,-14.07,"[[1, 3, 0.87], [1, 4, 0.87], [2, 4, 0.89], [1,..."
297,"[(1, 2), 0.66, (1, 3), 0.82, (2, 3), 0.37, (3,...","[[new_layer_p, 248, -0.79, 0.0, new_layer_p, 2...","[new_layer_p, 248, -0.79, 0.0, new_layer_p, 40...",1.000000,n_12_worker_unknown_pid_1258253_ts_24-09-10__1...,-23.74,"[[1, 2, 0.66], [1, 3, 0.82], [2, 3, 0.37], [3,..."
298,"[(1, 3), 0.1, (1, 4), 0.13, (1, 5), 0.15, (2, ...","[[new_layer_p, 33, -0.79, 0.0, new_layer_p, 53...","[new_layer_p, 32, -0.79, 0.0, new_layer_p, 53,...",0.871124,n_12_worker_unknown_pid_1233006_ts_24-09-09__0...,-15.48,"[[1, 3, 0.1], [1, 4, 0.13], [1, 5, 0.15], [2, ..."


## Saving

In [85]:
adapt_gpt_circ_out_fpath = 'data/adapt_gpt_n12_090724_d2_out.json'

In [86]:
with open(adapt_gpt_circ_out_fpath, 'w') as f:
    f.write(json.dumps(adapt_gpt_val_samples_list))